In [8]:
!pip install cloudscraper

In [9]:
import cloudscraper

In [62]:
scraper = cloudscraper.create_scraper(
    browser={
        'browser': 'chrome',
        'platform': 'windows',
        'mobile': False
    }
)

In [112]:
def normalizuj_menjac(tekst):
    tekst = tekst.lower()
    if 'automatski' in tekst and 'poluautomatski' in tekst:
        return 'Automatski/Poluautomatski'  # oba se pominju - nejasno
    elif 'poluautomatski' in tekst:
        return 'Poluautomatski'
    elif 'automatski' in tekst:
        return 'Automatski'
    elif 'manuelni' in tekst:
        return 'Manuelni'
    else:
        return 'Nepoznato'

In [117]:
def procitaj_oglas(oglas):
    recnik={}
    cena=oglas.find('span', attrs={'data-testid': 'globalAdCardPriceTestId'}).find('span').text.replace('€', '').strip()
    recnik['cena']=int(cena.split()[0].replace('.',''))
    atributi=oglas.find('div',attrs={'data-testid': 'adDescription'}).find_all('div')
    godiste_karoserija=atributi[0].text
    recnik['starost']=2026-int(godiste_karoserija[:4])
    recnik['karoserija']=godiste_karoserija.split()[1]
    recnik['kubikaza']=int(atributi[1].text.split('|')[1].replace('cm3','').strip())
    recnik['gorivo']=atributi[1].text.split('|')[0].strip()
    recnik['kilometraza']=int(atributi[2].text.replace('.','').replace('km',''))
    recnik['snaga_kw']=int(atributi[3].text.split()[0].replace('kW',''))
    recnik['snaga_ks']=int(atributi[3].text.split()[1].replace('(',''))
    recnik['menjac']=normalizuj_menjac(atributi[4].text)
    recnik['broj_vrata']=int(atributi[5].text.split(',')[0].split('/')[1][0])
    recnik['broj_sedista']=int(atributi[5].text.split(',')[1].strip()[0])
    return recnik
    
    

In [120]:
lista=[]
for i in range(1,76):
    url=f'https://www.polovniautomobili.com/auto-oglasi/poslednja24h?showOldNew=all&sort=basic&page={i}'
    page=scraper.get(url)
    soup=BeautifulSoup(page.text,'html.parser')
    ads=soup.find_all('article')
    for ad in ads:
        try:
            lista.append(procitaj_oglas(ad))
        except Exception as e:
            continue
    time.sleep(1)


In [123]:
import pandas as pd

In [127]:
df=pd.DataFrame(lista)
df.head()
df.to_csv('cars.csv')